In [3]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from implicit.cpu.als import AlternatingLeastSquares

c:\Users\JuanOrtizAlonso\business-aware-recommender-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_DIR = Path("../data")

PROCESSED_DIR = DATA_DIR / "processed"

ARTICLES_PATH = PROCESSED_DIR / "articles.parquet"
CUSTOMERS_PATH = PROCESSED_DIR / "customers.parquet"
INTERACTIONS_PATH = PROCESSED_DIR / "interactions.parquet"

In [5]:
articles = pd.read_parquet(ARTICLES_PATH)

customers = pd.read_parquet(CUSTOMERS_PATH)

interactions = pd.read_parquet(INTERACTIONS_PATH)

print(f"Articles: {articles.shape}")
print(f"Customers: {customers.shape}")
print(f"Interactions: {interactions.shape}")

Articles: (99604, 25)
Customers: (100000, 7)
Interactions: (10725535, 3)


In [6]:
user_ids = interactions["customer_id"].unique()
item_ids = interactions["article_id"].unique()

user_to_idx = {
    user: idx
    for idx, user in enumerate(user_ids)
}

item_to_idx = {
    item: idx
    for idx, item in enumerate(item_ids)
}

In [7]:
idx_to_user = {
    idx: user
    for user, idx in user_to_idx.items()
}

idx_to_item = {
    idx: item
    for item, idx in item_to_idx.items()
}

In [8]:
interactions["user_idx"] = (
    interactions["customer_id"]
    .map(user_to_idx)
)

interactions["item_idx"] = (
    interactions["article_id"]
    .map(item_to_idx)
)

In [9]:
user_item_matrix = csr_matrix(
    (
        interactions["interaction"].astype(np.float32),
        (
            interactions["user_idx"],
            interactions["item_idx"]
        )
    )
)

print(user_item_matrix.shape)

(100000, 99604)


In [10]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20,
    random_state=42
)

c:\Users\JuanOrtizAlonso\business-aware-recommender-system\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [12]:
item_user_matrix = user_item_matrix.T

model.fit(item_user_matrix)

c:\Users\JuanOrtizAlonso\business-aware-recommender-system\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.3286762237548828 seconds
  warnings.warn(
100%|██████████| 20/20 [00:29<00:00,  1.49s/it]


In [13]:
test_user = 0

ids, scores = model.recommend(
    userid=test_user,
    user_items=user_item_matrix[test_user],
    N=10
)

print(ids)
print(scores)

[10883 47471  1343  7797 54951 74719  3043  4907 36707   555]
[0.00945971 0.00613704 0.00581435 0.00576036 0.00525423 0.00512458
 0.00472041 0.00469766 0.00459311 0.00452892]


In [14]:
TOP_K = 100

In [15]:
recommendations = []

n_users = user_item_matrix.shape[0]

for user_idx in range(n_users):

    item_ids_rec, scores = model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=TOP_K
    )

    user_df = pd.DataFrame({
        "user_idx": user_idx,
        "item_idx": item_ids_rec,
        "score": scores
    })

    recommendations.append(user_df)

    if user_idx % 5000 == 0:
        print(
            f"Processed {user_idx:,} users"
        )

Processed 0 users
Processed 5,000 users
Processed 10,000 users


KeyboardInterrupt: 

In [ ]:
candidate_recommendations = pd.concat(
    recommendations,
    ignore_index=True
)

In [ ]:
candidate_recommendations["customer_id"] = (
    candidate_recommendations["user_idx"]
    .map(idx_to_user)
)

candidate_recommendations["article_id"] = (
    candidate_recommendations["item_idx"]
    .map(idx_to_item)
)

In [ ]:
product_columns = [
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name"
]

In [ ]:
candidate_recommendations = (
    candidate_recommendations
    .merge(
        articles[product_columns],
        on="article_id",
        how="left"
    )
)

In [ ]:
candidate_recommendations = (
    candidate_recommendations
    .sort_values(
        ["customer_id", "score"],
        ascending=[True, False]
    )
)

In [ ]:
with open(
    PROCESSED_DIR / "user_to_idx.pkl",
    "wb"
) as f:
    pickle.dump(user_to_idx, f)

with open(
    PROCESSED_DIR / "item_to_idx.pkl",
    "wb"
) as f:
    pickle.dump(item_to_idx, f)

with open(
    PROCESSED_DIR / "als_model.pkl",
    "wb"
) as f:
    pickle.dump(model, f)